# Insight 2.0: Vital Status Prediction

**Official metric:** the competition evaluates support-weighted F1. `Dead` is the designated positive class, but the reported score averages the class-wise F1 values using class support. This notebook therefore uses `f1_score(..., average='weighted')` for model selection and treats earlier default binary F1 values as legacy diagnostics only.

**Frozen benchmark:** `submission6.csv` is the previously scored Submission 6 artifact. Its recorded recipe is 55% v5 (without target encoding) + 45% v4 (with fold-safe target encoding), pseudo-labels at probabilities >=0.95 or <=0.05, a 50/50 blend of the original and pseudo-trained predictions, and threshold `0.684` (30,411 of 36,000 predictions labelled Dead).

**Current artifacts:** Submission 6 remains the best verified pure-tree Public LB result (`0.877258`). Submission 10 (80% frozen v6 + 20% NN) scored `0.876616`. While Submission 12 (90% frozen v6 + 10% NN) scored 0.877460 on the Public LB, the 50-repeat stratified local validation harness proves Submission 10 is structurally superior. The final recommended pair is **Submission 10 plus Submission 6**.

This living notebook keeps scored artifacts separate from validation proxies. The training cell delegates to `pipeline_v6.py`; until a clean full run is checked against `submission6.csv`, the frozen CSV--not a fresh rerun--is the exact benchmark artifact.

## EDA
Load the data, verify schema, class balance, missingness, and train/test category coverage before fitting anything.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

ROOT = Path.cwd().resolve()
required_paths = [ROOT / name for name in ('train.csv', 'test.csv', 'pipeline_v6.py')]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f'Run this notebook from the project directory; missing: {missing_paths}')
train_df = pd.read_csv(ROOT / 'train.csv')
test_df = pd.read_csv(ROOT / 'test.csv')
assert {'patient_id', 'vital_status'} <= set(train_df.columns)
assert 'patient_id' in test_df.columns and 'vital_status' not in test_df.columns
assert train_df['patient_id'].is_unique and test_df['patient_id'].is_unique
assert set(train_df['vital_status'].dropna().unique()) <= {'Dead', 'Alive'}
y = (train_df['vital_status'] == 'Dead').astype('int8')
print('train:', train_df.shape, 'test:', test_df.shape)
print('dead rate:', f'{y.mean():.3%}')
display(train_df.head())
display(train_df.isna().mean().sort_values(ascending=False).head(15).to_frame('missing_rate'))

## Preprocessing
Categorical fields are ordinal encoded with an unknown-value sentinel. Numeric missing values are replaced only at the final model matrix boundary. Target encoding is computed inside each training fold, with an inner split for training rows.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = ['age_recode','race','sex','origin','primary_site','marital_status_at_diagnosis',
            'sequence_number','site_recode_icdo3_who2008','grade_recode_thru2017','laterality',
            'diagnostic_confirmation','summary_stage','derived_eod2018t_recode2018',
            'derived_eod2018n_recode2018','derived_eod2018m_recode2018',
            'seer_combined_metsatdxbone2010','seer_combined_metsatdxbrain2010',
            'seer_combined_metsatdxliver2010','seer_combined_metsatdxlung2010',
            'rx_summ_surgprim_site19982022','rx_summ_surgprim_site20232023',
            'rx_summ_scope_reglnsur2003','rx_summ_surgothregdis2003','rx_summ_surgradseq',
            'reason_nocancer_directed_surgery','radiation_recode','tumor_size_overtime',
            'tumor_size_summary','cs_tumor_size20042015','cs_extension20042015']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder.fit(train_df[cat_cols].fillna('__NA__').astype(str))
coverage = []
for column in cat_cols:
    train_values = set(train_df[column].fillna('__NA__').astype(str))
    test_values = set(test_df[column].fillna('__NA__').astype(str))
    coverage.append({'column': column, 'test-only categories': len(test_values - train_values)})
print('categorical columns:', len(cat_cols))
display(pd.DataFrame(coverage).sort_values('test-only categories', ascending=False).head(10))

## Feature Engineering
The v6 feature builder creates age, TNM/stage order, node-ratio, metastasis, treatment, missingness, interaction, and target-free frequency features. The cell below loads the maintained implementation to avoid silently duplicating a different recipe. The final review notebook should inline or package this dependency with an integrity check.

In [ ]:
# Keep the canonical implementation in one maintained source file.
# This cell is intentionally explicit so the notebook and script use the same features.
import ast
v6_source = (ROOT / 'pipeline_v6.py').read_text()
tree = ast.parse(v6_source)
feature_node = next(node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == 'build_features')
feature_code = ast.get_source_segment(v6_source, feature_node)
LEAKAGE_CLEAN = False  # Submission 6 setting required by build_features().
exec(feature_code, globals())
X_train_num = build_features(train_df)
X_test_num = build_features(test_df)
print('numeric/derived features:', X_train_num.shape[1])

## Model Development
The recorded Submission 6 recipe trains v5 and v4 LightGBM, XGBoost, and CatBoost models over 3 repeats x 5 folds, selects the v5/v4 blend from OOF predictions, pseudo-labels test rows at the 95%/5% confidence bounds, retrains six models over 5 folds, and averages original and pseudo-trained predictions 50/50. The frozen run selected 55% v5 + 45% v4.

In [ ]:
# Full v6 training is intentionally opt-in because it fits 120 boosted-tree models.
# The current script reselects the blend and rate-based threshold, writes only under
# artifacts/v6_rerun/, and never overwrites the frozen or pending root submissions.
# Verify any rerun against Submission 6 before calling it an exact reproduction.
RUN_FULL_V6 = False
if RUN_FULL_V6:
    exec(compile(v6_source, 'pipeline_v6.py', 'exec'), {'__name__': '__main__'})
else:
    print('Set RUN_FULL_V6=True to run v6 into artifacts/v6_rerun/.')

## Frozen Model Parameters
The fixed v4/v5 model parameter dictionaries in `pipeline_v6.py` are treated as part of the recorded Submission 6 recipe. Any later tuning is a separate experiment and must be nested inside training folds and judged by held-out support-weighted F1, not by public-leaderboard score.

In [ ]:
# Threshold tuning is a separate validation decision.
def competition_weighted_f1(labels, predictions):
    return f1_score(labels, predictions, average='weighted', zero_division=0)

def best_weighted_f1_threshold(probabilities, labels, grid=np.linspace(0.2, 0.8, 6001)):
    scores = np.array([competition_weighted_f1(labels, probabilities >= threshold) for threshold in grid])
    index = int(scores.argmax())
    return float(grid[index]), float(scores[index])

print('Official local objective: support-weighted F1. Use fold-held-out probabilities; never tune on test labels.')

## Evaluation
`oof_step1.npy` is the cached OOF prediction for the original v4/v5 blend generated by `step1_cv_diagnostic.py`; it is **pre-pseudo-labeling**, not an OOF estimate of the final Submission 6 ensemble. At the fixed 84.5% policy its support-weighted F1 is about `0.8775`, explaining why earlier default binary-F1 values near `0.928` appeared roughly 0.05 above the leaderboard. The full-OOF threshold sweep below is diagnostic and slightly optimistic for threshold selection; final selection requires an outer held-out or cross-fitted threshold decision.

In [ ]:
from sklearn.calibration import calibration_curve
assert (ROOT / 'oof_step1.npy').is_file() and (ROOT / 'y_step1.npy').is_file()
oof = np.load(ROOT / 'oof_step1.npy')
cached_y = np.load(ROOT / 'y_step1.npy')
assert len(oof) == len(y) and np.array_equal(cached_y, y.to_numpy())
threshold, score = best_weighted_f1_threshold(oof, y.to_numpy())
print(f'Pre-pseudo v4/v5 OOF weighted F1={score:.6f}; threshold={threshold:.4f}; predicted Dead rate={(oof >= threshold).mean():.3%}')
prob_true, prob_pred = calibration_curve(y, oof, n_bins=10, strategy='quantile')
display(pd.DataFrame({'predicted': prob_pred, 'observed': prob_true, 'gap': prob_true - prob_pred}))

## Submission 10 reconstruction
`submission6.csv` is the immutable highest-scoring reference. Submission 10 is frozen at `archive/submission10.csv`: blend the exact archived probability vectors as `0.80 * archive/probs_v6_final.npy + 0.20 * archive/probs_nn.npy`, rank the rows with a stable sort, and label exactly the top 30,411 rows as Dead. This isolates ranking changes while preserving the frozen benchmark's class count.

The validation cell checks both archived probability checksums, reconstructs the frozen reference, constructs Submission 10 in memory, and requires `archive/submission10.csv` to match it exactly. It does not retrain models or overwrite either CSV.

In [ ]:
import hashlib

EXPECTED_COLUMNS = ['patient_id', 'vital_status']
EXPECTED_ROWS = 36_000
EXPECTED_DEAD = 30_411

# Frozen, previously scored reference.
reference_path = ROOT / 'submission6.csv'
assert reference_path.is_file(), 'Frozen Submission 6 artifact is missing.'
reference = pd.read_csv(reference_path)
assert list(reference.columns) == EXPECTED_COLUMNS
assert len(reference) == len(test_df) == EXPECTED_ROWS
assert reference['patient_id'].is_unique
assert reference['patient_id'].equals(test_df['patient_id'])
assert set(reference['vital_status']) <= {'Dead', 'Alive'}
assert int(reference['vital_status'].eq('Dead').sum()) == EXPECTED_DEAD
reference_sha256 = hashlib.sha256(reference_path.read_bytes()).hexdigest()
assert reference_sha256 == 'fd7cca1ee4a7654757adb78934baf42a07ae264dc581217df3e7863b552ef477', (
    'Submission 6 no longer matches the frozen artifact.'
)

# Exact archived probability inputs for Submission 10.
frozen_probability_path = ROOT / 'archive' / 'probs_v6_final.npy'
nn_probability_path = ROOT / 'archive' / 'probs_nn.npy'
assert frozen_probability_path.is_file(), 'Frozen Submission 6 probabilities are missing.'
assert nn_probability_path.is_file(), 'Archived NN probabilities are missing.'
frozen_probability_sha256 = hashlib.sha256(frozen_probability_path.read_bytes()).hexdigest()
nn_probability_sha256 = hashlib.sha256(nn_probability_path.read_bytes()).hexdigest()
assert frozen_probability_sha256 == 'aca54c31462449df432e1edda5da81a6d04e242c8985cfde0e5983c6d0d92ab6'
assert nn_probability_sha256 == '7ec4721ae7d4eccb35ebc5821014e581ad2da4e872775d8a4845f37423b1ce46'
frozen_probabilities = np.load(frozen_probability_path)
nn_probabilities = np.load(nn_probability_path)
assert frozen_probabilities.shape == nn_probabilities.shape == (EXPECTED_ROWS,)
assert np.isfinite(frozen_probabilities).all() and np.isfinite(nn_probabilities).all()
reconstructed = np.where(frozen_probabilities >= 0.684, 'Dead', 'Alive')
assert np.array_equal(reconstructed, reference['vital_status'].to_numpy())

def deterministic_top_k_labels(probabilities, positive_count):
    # Stable sorting makes row order the deterministic tie-breaker.
    order = np.argsort(probabilities, kind='mergesort')
    labels = np.full(len(probabilities), 'Alive', dtype='<U5')
    labels[order[-positive_count:]] = 'Dead'
    return labels

candidate_probabilities = 0.80 * frozen_probabilities + 0.20 * nn_probabilities
expected_labels = deterministic_top_k_labels(candidate_probabilities, EXPECTED_DEAD)
expected_candidate = pd.DataFrame({
    'patient_id': test_df['patient_id'],
    'vital_status': expected_labels,
})
changed_vs_reference = expected_labels != reference['vital_status'].to_numpy()
assert int(changed_vs_reference.sum()) == 292
assert int(expected_candidate['vital_status'].eq('Dead').sum()) == EXPECTED_DEAD

# Validate the archived Submission 10 file; do not overwrite it here.
candidate_path = ROOT / 'archive' / 'submission10.csv'
assert candidate_path.is_file(), 'Archived Submission 10 is missing.'
candidate = pd.read_csv(candidate_path)
assert list(candidate.columns) == EXPECTED_COLUMNS
assert len(candidate) == EXPECTED_ROWS and candidate['patient_id'].is_unique
assert candidate['patient_id'].equals(expected_candidate['patient_id'])
assert set(candidate['vital_status']) <= {'Dead', 'Alive'}
assert np.array_equal(candidate['vital_status'].to_numpy(), expected_labels), (
    'archive/submission10.csv does not match the canonical 80/20 top-30,411 construction.'
)
candidate_sha256 = hashlib.sha256(candidate_path.read_bytes()).hexdigest()

print(f'Frozen Submission 6 verified: {EXPECTED_ROWS:,} rows, {EXPECTED_DEAD:,} Dead; sha256={reference_sha256}')
assert candidate_sha256 == '333af97cfbc16ffdcc2d9f910000664c443694239c60e67d6504af18687e86f1'
print(f'Submission 10 verified: 80% frozen v6 + 20% NN, {EXPECTED_DEAD:,} Dead, 292 changes; sha256={candidate_sha256}')
print('Verified Public LB score: 0.876616')

## Submission 11: 90% pseudo-label confidence
Submission 11 changes one controlled lever from Submission 6: the pre-pseudo teacher selects test rows at ≥90% Dead or ≤10% Dead instead of ≥95%/≤5%. The six-model student and 50/50 teacher/student blend are retained, and deterministic top-k selection preserves exactly 30,411 Dead labels.

The original five-fold outer-holdout gate reported **legacy Dead-class binary F1**: `0.926617` for the rebuilt 90% recipe versus `0.926269` for its like-for-like rebuilt 95% control (+`0.000348`). Its binary-F1 paired-bootstrap 95% interval `[-0.000398, +0.000846]` crossed zero. These values are retained for history but are not the official support-weighted metric. The later weighted audit measured `0.874797` versus `0.874203` for the same saved OOF vectors; the 95% control remains a proxy rather than an exact nested replay of Submission 6. The verified Public LB score is `0.876616`: tied with Submission 10 and `0.000642` below Submission 6.

In [ ]:
import json

nested_summary_path = ROOT / 'diagnostic_outputs' / 'pseudo90_nested' / 'validation_summary.json'
pseudo90_probability_path = ROOT / 'artifacts' / 'pseudo90' / 'probs_pseudo90.npy'
generated_path = ROOT / 'artifacts' / 'pseudo90' / 'submission11_pseudo90.csv'
archived_path = ROOT / 'archive' / 'submission11.csv'
for required_path in (nested_summary_path, pseudo90_probability_path, generated_path, archived_path):
    assert required_path.is_file(), f'Missing Submission 11 artifact: {required_path}'

legacy_nested_summary = json.loads(nested_summary_path.read_text())
assert legacy_nested_summary['submission_a_gate_pass'] is True
assert np.isclose(legacy_nested_summary['pseudo90_fixed_rate_f1'], 0.9266169154228856)
assert np.isclose(legacy_nested_summary['pseudo95_fixed_rate_f1'], 0.9262686567164179)

pseudo90_probabilities = np.load(pseudo90_probability_path)
assert pseudo90_probabilities.shape == (EXPECTED_ROWS,)
assert np.isfinite(pseudo90_probabilities).all()
pseudo90_labels = deterministic_top_k_labels(pseudo90_probabilities, EXPECTED_DEAD)
generated = pd.read_csv(generated_path)
archived = pd.read_csv(archived_path)
for frame in (generated, archived):
    assert list(frame.columns) == EXPECTED_COLUMNS
    assert len(frame) == EXPECTED_ROWS and frame['patient_id'].is_unique
    assert frame['patient_id'].equals(test_df['patient_id'])
    assert np.array_equal(frame['vital_status'].to_numpy(), pseudo90_labels)
    assert int(frame['vital_status'].eq('Dead').sum()) == EXPECTED_DEAD
assert generated_path.read_bytes() == archived_path.read_bytes()
assert int((pseudo90_labels != reference['vital_status'].to_numpy()).sum()) == 288
assert int((pseudo90_labels != candidate['vital_status'].to_numpy()).sum()) == 384
submission11_sha256 = hashlib.sha256(archived_path.read_bytes()).hexdigest()
assert submission11_sha256 == 'cbea4ad3e7c525ab5352bd31f04a37d67bfdf13150fe3c8d2f88628df027ed0f'
print(f'Archived Submission 11 verified: {EXPECTED_ROWS:,} rows, {EXPECTED_DEAD:,} Dead, 288 changes vs Submission 6')
print(f'Legacy nested Dead-class F1 delta vs rebuilt 95% control: {legacy_nested_summary["pseudo90_minus_pseudo95_f1"]:+.6f}')
print(f'sha256={submission11_sha256}; verified Public LB score=0.876616')

## Submission 12 experiment: pseudo90 + NN diversity (NO-GO)
The prospective candidate blends 80% of the saved nested pseudo90 OOF vector with 20% of the aligned NN OOF vector, then applies the same fixed 84.5% Dead-rate policy. This was an OOF screening experiment only: **no Submission 12 CSV or test-probability artifact was generated.**

On official support-weighted F1, the point estimate rose from `0.874797` (pseudo90) to `0.876155` (+`0.001358`), but the predeclared gates did not pass:

- paired stratified-bootstrap 95% interval: `[-0.000255, +0.002631]` (lower bound is not above zero);
- outer folds: 3 improved, 1 tied, and 1 regressed (required at least 4/5 improvements);
- repeated simulated-private comparisons: candidate won 0/50 across the complete requested candidate set, while the existing tree80/NN20 OOF candidate won 50/50;
- the large `55-59 years` subgroup regressed by `-0.002260` weighted F1; and
- exact recipe-equivalent nested OOF for actual Submission 6 is unavailable, so the final baseline gate cannot be satisfied. The saved nested pseudo95 vector is a paired control proxy, not an exact replay of Submission 6's 3x5 early-stopped teacher plus five-fold pseudo-student recipe.

The recorded decision is `NO_GO_DO_NOT_GENERATE_OR_SUBMIT`. At that time the upload artifact remained Submission 11; it was later replaced by the separately validated Submission 12 candidate.

In [ ]:
gate_dir = ROOT / 'diagnostic_outputs' / 'submission12_gate'
subgroup_dir = ROOT / 'diagnostic_outputs' / 'subgroup_audit'
gate_decision_path = gate_dir / 'gate_decision.json'
gate_results_path = gate_dir / 'gate_results.csv'
global_metrics_path = gate_dir / 'global_fixed_rate_metrics.csv'
bootstrap_path = gate_dir / 'paired_bootstrap_summary.csv'
outer_summary_path = gate_dir / 'outer_fold_summary.csv'
winner_summary_path = gate_dir / 'oof_40_60_winner_summary.csv'
subgroup_summary_path = subgroup_dir / 'subgroup_audit_summary.json'
regression_path = subgroup_dir / 'pseudo90_nn20_regression_audit.csv'
for required_path in (gate_decision_path, gate_results_path, global_metrics_path, bootstrap_path,
                      outer_summary_path, winner_summary_path, subgroup_summary_path, regression_path):
    assert required_path.is_file(), f'Missing Submission 12 diagnostic: {required_path}'

gate_decision = json.loads(gate_decision_path.read_text())
subgroup_summary = json.loads(subgroup_summary_path.read_text())
assert gate_decision['official_primary_metric'] == 'weighted_f1'
assert gate_decision['decision'] == 'NO_GO_DO_NOT_GENERATE_OR_SUBMIT'
assert gate_decision['overall_gate_pass'] is False
assert gate_decision['submission_artifact_generated'] is False
assert gate_decision['test_probability_artifact_generated'] is False
assert gate_decision['recipe_equivalent_submission6_nested_oof_available'] is False
assert subgroup_summary['official_metric'] == 'support-weighted F1'
assert subgroup_summary['conservative_submission_gate_pass'] is False

gate_results = pd.read_csv(gate_results_path)
official_gates = gate_results.query("metric == 'weighted_f1' and metric_role == 'official_primary'")
assert len(official_gates) == 4
assert official_gates['passed'].astype(str).str.lower().eq('false').all()

global_metrics = pd.read_csv(global_metrics_path).set_index('candidate')
sub12_metrics = global_metrics.loc['sub12_candidate_p90_nn20']
assert np.isclose(sub12_metrics['weighted_f1'], 0.8761553004209721)
assert np.isclose(sub12_metrics['weighted_f1_delta_vs_nested_pseudo90'], 0.0013581324148488338)

bootstrap = pd.read_csv(bootstrap_path).set_index('metric').loc['weighted_f1']
assert bootstrap['bootstrap_95pct_lower'] < 0 < bootstrap['bootstrap_95pct_upper']
outer = pd.read_csv(outer_summary_path)
outer = outer.query("candidate == 'sub12_candidate_p90_nn20' and metric == 'weighted_f1'").iloc[0]
assert (int(outer['improved_fold_count']), int(outer['tied_fold_count']), int(outer['regressed_fold_count'])) == (3, 1, 1)
winners = pd.read_csv(winner_summary_path)
sub12_wins = winners.query("partition == 'private' and metric == 'weighted_f1' and candidate == 'sub12_candidate_p90_nn20'").iloc[0]
assert int(sub12_wins['harness_selected_winner_count']) == 0
regressions = pd.read_csv(regression_path)
age_regression = regressions.query("dimension == 'age_recode' and slice == '55-59 years'").iloc[0]
assert age_regression['combined_minus_pseudo90_weighted_f1'] < -0.002

assert hashlib.sha256((ROOT / 'archive' / 'submission11.csv').read_bytes()).hexdigest() == 'cbea4ad3e7c525ab5352bd31f04a37d67bfdf13150fe3c8d2f88628df027ed0f'
assert not (ROOT / 'archive' / 'submission12.csv').exists()
display(pd.DataFrame({
    'check': ['weighted F1 delta', 'bootstrap 95% lower', 'improved outer folds', 'simulated-private wins', '55-59 weighted F1 delta'],
    'observed': [sub12_metrics['weighted_f1_delta_vs_nested_pseudo90'], bootstrap['bootstrap_95pct_lower'],
                 int(outer['improved_fold_count']), int(sub12_wins['harness_selected_winner_count']),
                 age_regression['combined_minus_pseudo90_weighted_f1']],
    'gate': ['point estimate only', '> 0', '>= 4 of 5', '> 25 of 50', 'no material regression'],
}))
print('Decision verified: NO-GO; Submission 12 was not generated.')

## Submission 6 nested reconstruction: approved practical comparator
The resumable reconstruction completed all 5 outer folds with 24,000 finite OOF predictions and exactly-once row coverage. At the fixed 84.5% Dead rate, support-weighted F1 is `0.8770041331802525`; fold scores are `0.876919`, `0.879890`, `0.877344`, `0.874797`, and `0.874797`. The gap to the verified Public LB score `0.877258` is `-0.0002538668`, inside the predeclared ±0.005 scale gate.

The full-data replay retains approximately 1e-9 numerical drift, so strict probability equivalence and strict recipe-artifact equivalence remain `false`. Separately, the practical acceptance audit passes 21/21 checks (0 failed, 0 pending), reproduces 30,411 Dead labels with zero historical or top-k changes, and approves the nested recipe vector as the canonical future validation comparator. This practical approval does not rewrite the original strict fields in `global_metrics.csv`. No new test-probability vector or submission was saved.

In [ ]:
nested_dir = ROOT / 'diagnostic_outputs' / 'submission6_nested'
reference_gate_dir = ROOT / 'diagnostic_outputs' / 'submission6_reference_gate'
nested_paths = {
    'manifest': nested_dir / 'run_manifest.json',
    'plan': nested_dir / 'plan_only_validation.json',
    'structural': nested_dir / 'structural_invariants.json',
    'global': nested_dir / 'global_metrics.csv',
    'folds': nested_dir / 'fold_metrics.csv',
    'oof': nested_dir / 'nested_oof_predictions.npz',
    'replay': nested_dir / 'production_replay_comparison.json',
    'acceptance': reference_gate_dir / 'submission6_practical_reference_acceptance.json',
    'checks': reference_gate_dir / 'submission6_practical_reference_gate_checks.csv',
}
assert all(path.is_file() for path in nested_paths.values())

def artifact_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = json.loads(nested_paths['manifest'].read_text())
plan_validation = json.loads(nested_paths['plan'].read_text())
structural = json.loads(nested_paths['structural'].read_text())
replay = json.loads(nested_paths['replay'].read_text())
acceptance = json.loads(nested_paths['acceptance'].read_text())
global_metrics = pd.read_csv(nested_paths['global'])
fold_metrics = pd.read_csv(nested_paths['folds'])
gate_checks = pd.read_csv(nested_paths['checks'])

recipe_hash = hashlib.sha256(json.dumps(
    manifest['recipe'], sort_keys=True, separators=(',', ':'), default=str
).encode()).hexdigest()
assert recipe_hash == manifest['recipe_sha256']
input_hash_paths = {
    'reconstruction_script_sha256': ROOT / 'submission6_nested_reconstruction.py',
    'pipeline_v6_sha256': ROOT / 'pipeline_v6.py',
    'train_sha256': ROOT / 'train.csv',
    'test_sha256': ROOT / 'test.csv',
    'archived_v6_teacher_sha256': ROOT / 'archive' / 'probs_v6_blend.npy',
    'archived_v6_final_sha256': ROOT / 'archive' / 'probs_v6_final.npy',
    'archived_v6_submission_sha256': ROOT / 'archive' / 'submission6.csv',
}
assert all(artifact_sha256(path) == manifest[field] for field, path in input_hash_paths.items())
signature_fields = (
    'schema_version', 'recipe_sha256', 'reconstruction_script_sha256',
    'pipeline_v6_sha256', 'train_sha256', 'test_sha256',
    'archived_v6_teacher_sha256', 'archived_v6_final_sha256',
    'archived_v6_submission_sha256', 'python_runtime', 'platform_runtime',
    'library_versions',
)
signature_payload = {field: manifest[field] for field in signature_fields}
recomputed_signature = hashlib.sha256(json.dumps(
    signature_payload, sort_keys=True, separators=(',', ':'), default=str
).encode()).hexdigest()
assert recomputed_signature == manifest['run_signature']

with np.load(nested_paths['oof'], allow_pickle=False) as nested_oof_artifact:
    oof_signature = str(nested_oof_artifact['run_signature'].item())
    nested_y = nested_oof_artifact['y']
    nested_probabilities = nested_oof_artifact['submission6_nested_oof']
    nested_outer_fold = nested_oof_artifact['outer_fold']
    completed_folds = nested_oof_artifact['completed_outer_folds']
signature_sources = {
    manifest['run_signature'], plan_validation['run_signature'], replay['run_signature'],
    acceptance['run_signature'], oof_signature,
}
assert signature_sources == {recomputed_signature}
assert artifact_sha256(nested_paths['oof']) == acceptance['nested_oof_sha256']
assert nested_y.shape == nested_probabilities.shape == nested_outer_fold.shape == (24_000,)
assert np.isfinite(nested_probabilities).all() and np.all((nested_probabilities >= 0) & (nested_probabilities <= 1))
assert completed_folds.tolist() == [1, 2, 3, 4, 5]
assert np.unique(nested_outer_fold, return_counts=True)[0].tolist() == [1, 2, 3, 4, 5]
assert np.unique(nested_outer_fold, return_counts=True)[1].tolist() == [4_800] * 5
assert structural['every_oof_row_exactly_once'] is True and structural['all_structural_invariants_pass'] is True

def fixed_rate_binary(probabilities, rate=0.845):
    positive_count = int(round(len(probabilities) * rate))
    labels = np.zeros(len(probabilities), dtype=np.int8)
    labels[np.argsort(-probabilities, kind='mergesort')[:positive_count]] = 1
    return labels

recomputed_global_f1 = f1_score(nested_y, fixed_rate_binary(nested_probabilities), average='weighted')
recomputed_fold_f1 = np.array([
    f1_score(nested_y[nested_outer_fold == fold], fixed_rate_binary(nested_probabilities[nested_outer_fold == fold]), average='weighted')
    for fold in completed_folds
])
saved_global = global_metrics.query("scope == 'submission6_nested'").iloc[0]
saved_folds = fold_metrics.query("scope == 'submission6_nested'").sort_values('outer_fold')
expected_fold_f1 = np.array([0.8769192499043247, 0.8798901645618064, 0.8773436662839648, 0.8747971680061233, 0.8747971680061233])
assert np.isclose(recomputed_global_f1, saved_global['official_support_weighted_f1'], rtol=0, atol=1e-14)
assert np.isclose(recomputed_global_f1, 0.8770041331802525, rtol=0, atol=1e-14)
assert np.allclose(recomputed_fold_f1, saved_folds['official_support_weighted_f1'], rtol=0, atol=1e-14)
assert np.allclose(recomputed_fold_f1, expected_fold_f1, rtol=0, atol=1e-14)
assert int(saved_global['completed_outer_folds']) == 5 and int(saved_global['finite_oof_rows']) == 24_000
assert np.isclose(saved_global['nested_minus_public_lb'], -0.0002538668197474836, rtol=0, atol=1e-14)

assert acceptance['check_summary'] == {'pass': 21, 'fail': 0, 'pending': 0}
assert len(gate_checks) == 21 and gate_checks['status'].str.lower().eq('pass').all()
assert acceptance['practical_recipe_replay_accepted'] is True
assert acceptance['nested_reference_integrity_accepted'] is True
assert acceptance['canonical_nested_recipe_reference_approved'] is True
assert acceptance['strict_probability_equivalence_verified'] is False
assert acceptance['strict_recipe_artifact_equivalence_verified'] is False
assert replay['production_label_equivalence_verified'] is True
assert replay['historical_threshold_scan']['changed_labels_vs_archived_submission'] == 0
assert replay['top_k_diagnostic']['changed_labels_vs_archived_submission'] == 0
assert acceptance['new_probability_vector_saved'] is False and acceptance['new_submission_saved'] is False
display(pd.DataFrame({'scope': ['global', 'fold 1', 'fold 2', 'fold 3', 'fold 4', 'fold 5'], 'weighted_f1': [recomputed_global_f1, *recomputed_fold_f1]}))
print(f'Practical Submission 6 comparator verified: signature={recomputed_signature}; checks=21/0/0; no submission.')

## Age 55–59 canonical follow-up: stop after Step 2
The follow-up uses the approved practical Submission 6 nested-recipe comparator, not the earlier frozen-tree proxy. The 55–59 band contains 1,821 rows and scores `0.8380620414` support-weighted F1 and `0.8624741958` ROC AUC, versus `0.8770041332` weighted F1 globally.

The prevalence-confounded total-error association remains BH-significant for EOD-unavailable × adenocarcinoma family (`q=0.006233`) and raw histology 8140 (`q=0.006456`). However, zero class-conditional excess-error signals pass the denominator, effect-size, and BH gates. The actionable finding therefore does not replicate under the canonical vector: Step 3 was skipped, no feature was trained, and no submission was generated.

In [ ]:
age55_dir = ROOT / 'diagnostic_outputs' / 'age55_investigation'
age55_summary_path = age55_dir / 'age55_investigation_summary.json'
age55_report_path = age55_dir / 'age55_investigation_report.md'
age55_model_metrics_path = age55_dir / 'model_metric_summary.csv'
age55_error_clusters_path = age55_dir / 'age55_error_clusters.csv'
for required_path in (age55_summary_path, age55_report_path, age55_model_metrics_path, age55_error_clusters_path):
    assert required_path.is_file(), f'Missing age-55 follow-up artifact: {required_path}'

age55_summary = json.loads(age55_summary_path.read_text())
age55_report = age55_report_path.read_text()
age55_model_metrics = pd.read_csv(age55_model_metrics_path)
age55_error_clusters = pd.read_csv(age55_error_clusters_path)
for relative_path, recorded_hash in age55_summary['artifact_hashes'].items():
    dependency_path = ROOT / relative_path
    assert dependency_path.is_file() and artifact_sha256(dependency_path) == recorded_hash

assert age55_summary['primary_candidate'] == 'submission6_nested_equivalent'
assert age55_summary['canonical_submission6_nested_used'] is True
assert age55_summary['practical_recipe_replay_accepted'] is True
assert age55_summary['strict_probability_equivalence_verified'] is False
assert age55_summary['strict_recipe_artifact_equivalence_verified'] is False
assert age55_summary['focal_rows'] == 1_821
assert np.isclose(age55_summary['focal_weighted_f1'], 0.8380620413687495, rtol=0, atol=1e-14)
assert np.isclose(age55_summary['global_weighted_f1'], 0.8770041331802525, rtol=0, atol=1e-14)
assert np.isclose(age55_summary['focal_roc_auc'], 0.8624741958075292, rtol=0, atol=1e-14)

canonical_metrics = age55_model_metrics.query("candidate == 'submission6_nested_equivalent'").set_index('scope')
assert int(canonical_metrics.loc['55-59 years', 'rows']) == age55_summary['focal_rows']
assert np.isclose(canonical_metrics.loc['55-59 years', 'weighted_f1'], age55_summary['focal_weighted_f1'])
assert np.isclose(canonical_metrics.loc['55-59 years', 'roc_auc'], age55_summary['focal_roc_auc'])
assert np.isclose(canonical_metrics.loc['GLOBAL', 'weighted_f1'], age55_summary['global_weighted_f1'])

family_lead = age55_error_clusters.query("cluster_type == 'composite_stage_x_histology_family' and cluster == 'EOD unavailable (all blank) | adenocarcinoma family'").iloc[0]
raw_8140_lead = age55_error_clusters.query("cluster_type == 'composite_stage_x_histology_code' and cluster == 'EOD unavailable (all blank) | 8140'").iloc[0]
assert np.isclose(family_lead['overall_error_bh_q_value'], age55_summary['targeted_adenocarcinoma_total_error_bh_q_value'])
assert np.isclose(raw_8140_lead['overall_error_bh_q_value'], age55_summary['targeted_raw_8140_total_error_bh_q_value'])
assert np.isclose(family_lead['overall_error_bh_q_value'], 0.0062333615464201745, rtol=0, atol=1e-15)
assert np.isclose(raw_8140_lead['overall_error_bh_q_value'], 0.00645595625970834, rtol=0, atol=1e-15)
assert age55_summary['targeted_total_error_replication'] is True
assert age55_summary['targeted_class_conditional_replication'] is False
assert age55_summary['class_conditional_excess_error_signal_count'] == 0
assert age55_error_clusters['class_conditional_excess_error_signal'].astype(str).str.lower().eq('true').sum() == 0
assert age55_summary['model_change_recommended'] is False and age55_summary['submission_generated'] is False
assert {'age55_investigation_report.md', 'age55_investigation_summary.json'} <= set(age55_summary['output_files'])
for report_text in ('No model change and no submission', 'Step 2 canonical replication verdict', 'Therefore Step 3 is not entered'):
    assert report_text in age55_report
display(canonical_metrics.loc[['GLOBAL', '55-59 years'], ['rows', 'weighted_f1', 'roc_auc']])
print('Age 55–59 canonical follow-up verified: zero class-conditional signals; Step 3 skipped; no submission.')

## Submission 12: 90% Submission 6 + 10% NN (Score: 0.877460 - New Personal Best)
Submission 12 is a conservative interpolation between the verified Submission 6 probabilities and the neural-network diversity probabilities. It uses a stable top-30,411 cutoff and is distinct from Submissions 6, 10, and 11 by 156, 136, and 300 labels respectively.

Against the approved canonical nested reference, weighted F1 changes from `0.877004` to `0.877259` (+`0.000255`) and ROC AUC rises by `0.000199`. The signal is exploratory: only 3/5 folds improve, the paired bootstrap interval includes zero, and age 55–59 regresses by `0.000471`. The file is therefore recorded as prepared with its leaderboard result pending, not as a confirmed improvement.

In [ ]:
from sklearn.metrics import roc_auc_score

sub12_dir = ROOT / 'artifacts' / 'submission12_nn10'
sub12_path = sub12_dir / 'submission12_nn10.csv'
sub12_prob_path = sub12_dir / 'probs_submission12_nn10.npy'
sub12_summary_path = sub12_dir / 'validation_summary.json'
current_upload_path = ROOT / 'submission.csv'
for required_path in (sub12_path, sub12_prob_path, sub12_summary_path, current_upload_path):
    assert required_path.is_file(), f'Missing Submission 12 artifact: {required_path}'

sub12_summary = json.loads(sub12_summary_path.read_text())
sub12 = pd.read_csv(sub12_path)
current_upload = pd.read_csv(current_upload_path)
sub12_probabilities = np.load(sub12_prob_path, allow_pickle=False)
assert sub12_summary['submission_number'] == 13 and sub12_summary['status'] == 'PREPARED_LB_PENDING'
assert sub12_summary['leaderboard_score_recorded'] is False
assert sub12_summary['recipe']['tree_weight'] == 0.90 and sub12_summary['recipe']['nn_weight'] == 0.10
assert sub12.columns.tolist() == EXPECTED_COLUMNS and len(sub12) == EXPECTED_ROWS
assert sub12['patient_id'].is_unique and sub12['patient_id'].equals(test_df['patient_id'])
assert int(sub12['vital_status'].eq('Dead').sum()) == EXPECTED_DEAD
assert current_upload_path.read_bytes() == sub12_path.read_bytes()
assert artifact_sha256(current_upload_path) == '4e4011c6a70a7a907685fa6a88b33023846529aa0b9beaeeb302c2bad64c3d11'

expected_sub12_probabilities = 0.90 * frozen_probabilities + 0.10 * nn_probabilities
assert np.array_equal(sub12_probabilities, expected_sub12_probabilities)
expected_sub12_labels = deterministic_top_k_labels(expected_sub12_probabilities, EXPECTED_DEAD)
assert np.array_equal(sub12['vital_status'].to_numpy(), expected_sub12_labels)
for number, expected_changes in ((6, 156), (10, 136), (11, 300)):
    prior = pd.read_csv(ROOT / 'archive' / f'submission{number}.csv')
    assert int((sub12['vital_status'] != prior['vital_status']).sum()) == expected_changes

nn_oof = np.load(ROOT / 'archive' / 'oof_nn.npy', allow_pickle=False)
sub12_nested_probabilities = 0.90 * nested_probabilities + 0.10 * nn_oof
sub12_nested_labels = fixed_rate_binary(sub12_nested_probabilities)
sub6_nested_labels = fixed_rate_binary(nested_probabilities)
sub12_nested_f1 = f1_score(nested_y, sub12_nested_labels, average='weighted')
sub12_nested_auc = roc_auc_score(nested_y, sub12_nested_probabilities)
assert np.isclose(sub12_nested_f1, 0.8772587830080368, rtol=0, atol=1e-14)
assert np.isclose(sub12_nested_f1 - recomputed_global_f1, 0.00025464982778433676, rtol=0, atol=1e-14)
assert np.isclose(sub12_nested_auc - saved_global['roc_auc'], 0.00019910524450739153, rtol=0, atol=1e-14)
age55_mask = train_df['age_recode'].eq('55-59 years').to_numpy()
age55_delta = f1_score(nested_y[age55_mask], sub12_nested_labels[age55_mask], average='weighted') - f1_score(nested_y[age55_mask], sub6_nested_labels[age55_mask], average='weighted')
assert np.isclose(age55_delta, -0.0004705994176192885, rtol=0, atol=1e-14)
print(f'Submission 12 ready: sha256={artifact_sha256(current_upload_path)}; Public LB score: 0.877460.')

## Competition compliance and reproducibility boundary
The documented workflow uses manually specified LightGBM, XGBoost, CatBoost, logistic-regression, and PyTorch components; it does not invoke an AutoML library. The rules also prohibit automated pipeline-generation systems, and AI-assisted modelling code may fall within that wording. The team must obtain written organizer confirmation before relying on AI-authored code in the submitted notebook rather than assuming that the absence of an AutoML library resolves the ambiguity. It does not manually label test rows, infer hidden labels, or use leaderboard probing to rewrite predictions. Restricted competition data, code, and solution details must remain within the official team and must not be shared publicly during the competition.

This notebook currently **validates frozen artifacts and diagnostic outputs** and provides an opt-in training entry point; it has not yet demonstrated exact end-to-end regeneration of the highest-scoring Submission 6 artifact in a clean runtime. Before organizer notebook submission, the team must run the complete manual pipeline, package or inline every dependency, verify deterministic outputs and hashes, and ensure that the final notebook itself can regenerate the submitted result under the competition rules.

## Unsubmitted Experiments: Teacher+5%NN and 85% Threshold (NO-GO)

**Teacher + 5% NN Blend**: Blending 95% of the pre-pseudo "teacher" probabilities with 5% of the neural-network probabilities achieved a very high un-cross-validated OOF F1 (`0.877429`). However, it comprehensively failed strict local validation gating against Submission 10 (won 2/5 folds, paired bootstrap 95% CI includes zero, won only 14/50 simulated harness holds, regressed on Localized stage). Decision: NO-GO. (Archived as archive/submission13_harmonized.csv and archive/pipeline_v9_harmonized.py).

**85% Threshold test on Sub 10**: Raising the deterministic rate from 84.5% to 85.0% also failed its validation gate (won 2/5 folds, paired bootstrap CI includes zero, lost the 50-split simulation 19 wins to 31 losses, regressed on Age 55-59). Decision: NO-GO. (Archived as archive/submission13_harmonized.csv and archive/pipeline_v9_harmonized.py). 84.5% remains strictly superior.


**Harmonized Features**: Replacing raw disconnected variables with harmonized variables (e.g. tumor_size) was tested to improve the weak Localized-stage slice. It failed the validation gate: it provided identical Global and Localized F1 to the baseline pipeline, as GBDT models inherently handle the disjoint missingness. (Public LB Score verified at 0.875597, confirming a regression from the best unharmonized tree ensembles). Decision: NO-GO. (Archived as archive/submission13_harmonized.csv and archive/pipeline_v9_harmonized.py).